# SCRAPPEO con sql

In [1]:
pip install requests beautifulsoup4

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\giovanni.cabrera\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## GENERAMOS LA BASE DE DATOS

In [6]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import random
from concurrent.futures import ThreadPoolExecutor

In [8]:
BASE_URL = "http://books.toscrape.com/catalogue/page-{}.html"
PAGINAS_A_SCRAPEAR = 50  # Puedes subirlo a 50 si quieres
AUTORES_FICTICIOS = ["J.R.R. Tolkien", "Agatha Christie", "Isaac Asimov", "Ursula K. Le Guin", "H.P. Lovecraft"]
CATEGORIAS_FICTICIAS = ["Fantasía", "Misterio", "Ciencia Ficción", "Clásicos"]

Aqui importamos lo que seria la base de datos en sql lite para hacerlo mas rapido

## GENERAMOS LAS TABLAS EN SQL 

In [9]:
def inicializar_db():
    conn = sqlite3.connect('mi_libreria.db')
    cursor = conn.cursor()
    cursor.execute('PRAGMA foreign_keys = ON;')
    
    # Crear Tablas
    cursor.execute('CREATE TABLE IF NOT EXISTS autores (id INTEGER PRIMARY KEY, nombre TEXT)')
    cursor.execute('CREATE TABLE IF NOT EXISTS categorias (id INTEGER PRIMARY KEY, nombre TEXT)')
    cursor.execute('''CREATE TABLE IF NOT EXISTS libros (
                        id INTEGER PRIMARY KEY, 
                        titulo TEXT, 
                        precio TEXT, 
                        autor_id INTEGER, 
                        cat_id INTEGER,
                        FOREIGN KEY(autor_id) REFERENCES autores(id),
                        FOREIGN KEY(cat_id) REFERENCES categorias(id))''')
    
    # Llenar autores y categorías si están vacías
    for nombre in AUTORES_FICTICIOS:
        cursor.execute('INSERT OR IGNORE INTO autores (nombre) VALUES (?)', (nombre,))
    for nombre in CATEGORIAS_FICTICIAS:
        cursor.execute('INSERT OR IGNORE INTO categorias (nombre) VALUES (?)', (nombre,))
    
    conn.commit()
    conn.close()

In [10]:
def scrapear_pagina(numero_pagina):
    url = BASE_URL.format(numero_pagina)
    print(f"🚀 Extrayendo página {numero_pagina}...")
    
    try:
        r = requests.get(url, timeout=10)
        soup = BeautifulSoup(r.text, 'html.parser')
        datos_libros = []
        
        for libro in soup.find_all('article', class_='product_pod'):
            titulo = libro.h3.a['title']
            precio = libro.find('p', class_='price_color').text
            # Asignamos ID de autor y categoría al azar (del 1 al 4)
            autor_id = random.randint(1, len(AUTORES_FICTICIOS))
            cat_id = random.randint(1, len(CATEGORIAS_FICTICIAS))
            datos_libros.append((titulo, precio, autor_id, cat_id))
            
        return datos_libros
    except Exception as e:
        print(f"❌ Error en página {numero_pagina}: {e}")
        return []

In [12]:
if __name__ == "__main__":
    inicializar_db()
    
    urls_indices = range(1, PAGINAS_A_SCRAPEAR + 1)
    lista_final = []

    # Usamos 5 hilos para ir rápido
    with ThreadPoolExecutor(max_workers=5) as executor:
        resultados = executor.map(scrapear_pagina, urls_indices)
        for pag in resultados:
            lista_final.extend(pag)

    # Guardar todo de un solo golpe
    conn = sqlite3.connect('mi_libreria.db')
    cursor = conn.cursor()
    cursor.executemany('INSERT INTO libros (titulo, precio, autor_id, cat_id) VALUES (?, ?, ?, ?)', lista_final)
    conn.commit()
    conn.close()

    print(f"\n✅ ¡Listo! Guardados {len(lista_final)} libros con sus autores y categorías.")

🚀 Extrayendo página 1...
🚀 Extrayendo página 2...
🚀 Extrayendo página 3...
🚀 Extrayendo página 4...
🚀 Extrayendo página 5...
🚀 Extrayendo página 6...
🚀 Extrayendo página 7...
🚀 Extrayendo página 8...
🚀 Extrayendo página 9...
🚀 Extrayendo página 10...
🚀 Extrayendo página 11...
🚀 Extrayendo página 12...
🚀 Extrayendo página 13...
🚀 Extrayendo página 14...
🚀 Extrayendo página 15...
🚀 Extrayendo página 16...
🚀 Extrayendo página 17...
🚀 Extrayendo página 18...
🚀 Extrayendo página 19...
🚀 Extrayendo página 20...
🚀 Extrayendo página 21...
🚀 Extrayendo página 22...
🚀 Extrayendo página 23...
🚀 Extrayendo página 24...
🚀 Extrayendo página 25...
🚀 Extrayendo página 26...
🚀 Extrayendo página 27...
🚀 Extrayendo página 28...
🚀 Extrayendo página 29...
🚀 Extrayendo página 30...
🚀 Extrayendo página 31...
🚀 Extrayendo página 32...
🚀 Extrayendo página 33...
🚀 Extrayendo página 34...
🚀 Extrayendo página 35...
🚀 Extrayendo página 36...
🚀 Extrayendo página 37...
🚀 Extrayendo página 38...
🚀 Extrayendo página 3

In [18]:
if __name__ == "__main__":
    inicializar_db()
    
    urls_indices = range(1, PAGINAS_A_SCRAPEAR + 1)
    lista_final = []

    # Usamos 5 hilos para ir rápido
    with ThreadPoolExecutor(max_workers=5) as executor:
        resultados = executor.map(scrapear_pagina, urls_indices)
        for pag in resultados:
            lista_final.extend(pag)

    # Guardar todo de un solo golpe
    conn = sqlite3.connect('mi_libreria.db')
    cursor = conn.cursor()
    cursor.executemany('INSERT INTO libros (titulo, precio, autor_id, cat_id) VALUES (?, ?, ?, ?)', lista_final)
    conn.commit()
    # conn.close()

    print(f"\n✅ ¡Listo! Guardados {len(lista_final)} libros con sus autores y categorías.")

🚀 Extrayendo página 1...
🚀 Extrayendo página 2...
🚀 Extrayendo página 3...
🚀 Extrayendo página 4...
🚀 Extrayendo página 5...
🚀 Extrayendo página 6...
🚀 Extrayendo página 7...
🚀 Extrayendo página 8...
🚀 Extrayendo página 9...
🚀 Extrayendo página 10...
🚀 Extrayendo página 11...
🚀 Extrayendo página 12...
🚀 Extrayendo página 13...
🚀 Extrayendo página 14...
🚀 Extrayendo página 15...
🚀 Extrayendo página 16...
🚀 Extrayendo página 17...
🚀 Extrayendo página 18...
🚀 Extrayendo página 19...
🚀 Extrayendo página 20...
🚀 Extrayendo página 21...
🚀 Extrayendo página 22...
🚀 Extrayendo página 23...
🚀 Extrayendo página 24...
🚀 Extrayendo página 25...
🚀 Extrayendo página 26...
🚀 Extrayendo página 27...
🚀 Extrayendo página 28...
🚀 Extrayendo página 29...
🚀 Extrayendo página 30...
🚀 Extrayendo página 31...
🚀 Extrayendo página 32...
🚀 Extrayendo página 33...
🚀 Extrayendo página 34...
🚀 Extrayendo página 35...
🚀 Extrayendo página 36...
🚀 Extrayendo página 37...
🚀 Extrayendo página 38...
🚀 Extrayendo página 3

```text
        AUTORES                         CATEGORIAS
    +------------+                  +----------------+
    | id (PK)    |<---+        +--->| id (PK)        |
    | nombre     |    |        |    | nombre         |
    +------------+    |        |    +----------------+
                      |        |
                      |        |
                +-----|--------|-----+
                |      LIBROS        |
                +--------------------+
                | id (PK)            |
                | titulo             |
                | precio             |
                | autor_id (FK) -----+--> (Apunta a Autores)
                | cat_id (FK)  ------+--> (Apunta a Categorias)
                +--------------------+

In [22]:
cursor.execute("""
SELECT libros.titulo, autores.nombre, categorias.nombre, libros.precio
    FROM libros
    JOIN autores ON libros.autor_id = autores.id
    JOIN categorias ON libros.cat_id = categorias.id
""")
for fila in cursor.fetchall():
    print(f"|{fila[0]:<50} | {fila[1]:<20} | {fila[2]:<15} | {fila[3]:>10}")

|A Light in the Attic                               | Isaac Asimov         | Ciencia Ficción |    Â£51.77
|Tipping the Velvet                                 | J.R.R. Tolkien       | Clásicos        |    Â£53.74
|Soumission                                         | Ursula K. Le Guin    | Clásicos        |    Â£50.10
|Sharp Objects                                      | Agatha Christie      | Fantasía        |    Â£47.82
|Sapiens: A Brief History of Humankind              | Isaac Asimov         | Clásicos        |    Â£54.23
|The Requiem Red                                    | H.P. Lovecraft       | Misterio        |    Â£22.65
|The Dirty Little Secrets of Getting Your Dream Job | Agatha Christie      | Fantasía        |    Â£33.34
|The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull | J.R.R. Tolkien       | Misterio        |    Â£17.93
|The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics | Isaac Asimov     

In [ ]:
cursor.execute("""SELECT * from libros l join autores a on l.autor_id = a.id limit 10;""")
for fila in cursor.fetchall():
    print(fila)


(1, 'A Light in the Attic', 'Â£51.77', 3, 3, 3, 'Isaac Asimov')
(2, 'Tipping the Velvet', 'Â£53.74', 1, 4, 1, 'J.R.R. Tolkien')
(3, 'Soumission', 'Â£50.10', 4, 4, 4, 'Ursula K. Le Guin')
(4, 'Sharp Objects', 'Â£47.82', 2, 1, 2, 'Agatha Christie')
(5, 'Sapiens: A Brief History of Humankind', 'Â£54.23', 3, 4, 3, 'Isaac Asimov')
(6, 'The Requiem Red', 'Â£22.65', 5, 2, 5, 'H.P. Lovecraft')
(7, 'The Dirty Little Secrets of Getting Your Dream Job', 'Â£33.34', 2, 1, 2, 'Agatha Christie')
(8, 'The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull', 'Â£17.93', 1, 2, 1, 'J.R.R. Tolkien')
(9, 'The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics', 'Â£22.60', 3, 2, 3, 'Isaac Asimov')
(10, 'The Black Maria', 'Â£52.15', 4, 3, 4, 'Ursula K. Le Guin')
(11, 'Starving Hearts (Triangular Trade Trilogy, #1)', 'Â£13.99', 4, 4, 4, 'Ursula K. Le Guin')
(12, "Shakespeare's Sonnets", 'Â£20.66', 4, 2, 4, 'Ursula K. Le Guin')
(13, 'Set M